# Joint QCD histograms (cross-section weighted)

Joint, cross-section-weighted histograms over all QCD HT slices in
`datasets.json`. Each slice is weighted by `cross_section / N_original`, where
`N_original` is the original event count from the **first bin of the `cutflow`**
histogram, summed over the slice's files.

Contents: masses (leading jet, leading dijet, all-jet system), leading pT and
HT for the scouting jets / GenJets / GenParts, and pre- vs post-JEC comparisons
on the scouting jets.

Reading: one `NanoEventsFactory.from_root(..., mode="dask")` call **per slice**
(multi-file dict requires `mode="dask"`); arrays are materialized once per slice
with `dask.compute`, and the weight rides in `events.metadata["weight"]`.

Prereqs:
1. `python scripts/make_dataset_json.py <eos_path> -o datasets.json`
2. Cross sections from `run3-mj-pass-the-aux/mj_samples_xs.json` (set `XS_JSON`).

In [ ]:
import sys, pathlib, json

# Make the package importable without `pip install -e .` (src/ layout).
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))

import numpy as np
import awkward as ak
import uproot
import dask
import hist
import vector
import matplotlib.pyplot as plt
from coffea.nanoevents import NanoEventsFactory, BaseSchema

from run3_mj_analyzer.fileset import load_fileset

vector.register_awkward()  # enables Momentum4D behaviors (.mass, .px, ...)

In [ ]:
JSON_PATH = "../datasets.json"  # written by scripts/make_dataset_json.py

fileset = load_fileset(JSON_PATH)
print(f"{len(fileset)} slices:")
for name, ds in fileset.items():
    print(f"  {name}: {len(ds['files'])} files")

## Cross sections

Loaded from the shared aux repo `run3-mj-pass-the-aux/mj_samples_xs.json`
(`{dataset: {"xs_pb": ..., "sig_pb": ...}}`); we use `xs_pb`. The check below
raises if any slice in the fileset is missing from that file.

In [ ]:
# Path to the cross-section JSON in the run3-mj-pass-the-aux checkout.
# Default assumes it sits next to run3-mj-analyzer; adjust if yours differs.
XS_JSON = "../../run3-mj-pass-the-aux/mj_samples_xs.json"

with open(XS_JSON) as f:
    _xs = json.load(f)
CROSS_SECTIONS = {name: info["xs_pb"] for name, info in _xs.items()}

missing = [d for d in fileset if d not in CROSS_SECTIONS]
if missing:
    raise ValueError(
        f"No cross section in {XS_JSON} for these slices:\n  "
        + "\n  ".join(missing)
    )
print(f"Loaded {len(CROSS_SECTIONS)} cross sections from {XS_JSON}")

## Helpers and per-slice weight

`weight = cross_section / N_original` (optionally scaled by an integrated
luminosity). `N_original` is summed over the slice's files from the first
`cutflow` bin (read with uproot -- the cutflow is not in the events tree).

In [ ]:
LUMI = 1.0  # pb^-1. Leave 1.0 for pure xsec/N weights; set to lumi for yields.


def jet_p4(events):
    """Per-event Momentum4D scouting jets (corrected pt/m) from ScoutingPFJet."""
    if "ScoutingPFJet" in events.fields:
        j = events["ScoutingPFJet"]
        pt, eta, phi, m = j.pt, j.eta, j.phi, j.m
    else:  # flat fallback
        pt = events["ScoutingPFJet_pt"]
        eta = events["ScoutingPFJet_eta"]
        phi = events["ScoutingPFJet_phi"]
        m = events["ScoutingPFJet_m"]
    return ak.zip(
        {"pt": pt, "eta": eta, "phi": phi, "mass": m}, with_name="Momentum4D"
    )


def lead_pt(pt):
    """Highest pT per event (events with >=1 object). Order-independent, so it is
    robust to JEC re-ordering of the stored jets."""
    return ak.max(pt[ak.num(pt) >= 1], axis=1)


def original_event_count(filepath):
    """Original events read for a file = first bin of its 'cutflow' histogram."""
    with uproot.open(filepath) as f:
        cutflow = f["cutflow"]
        try:
            return float(cutflow.values()[0])
        except Exception:
            return float(cutflow.to_hist().values()[0])


def slice_n_original(filepaths):
    """Sum the first cutflow bin over every file in a slice (cheap: no tree read)."""
    return sum(original_event_count(fp) for fp in filepaths)

## Fill all histograms (one `from_root` call per slice)

Every observable is built lazily for the slice and materialized in a single
`dask.compute`. Mass histograms keep a `dataset` category axis (for stacking);
the pT/HT/JEC comparison histograms are plain 1D weighted hists summed over
slices. `dask.compute` uses the default scheduler -- no cluster needed.

In [ ]:
DATASETS = list(fileset)
N_FILES = None  # set an int to limit files/slice while testing (under-fills,
                # but the weight still uses the FULL slice N_original)

# --- mass histograms (stacked over slices) ---
cat = hist.axis.StrCategory(DATASETS, name="dataset", label="QCD slice")
h_lead = hist.Hist(cat, hist.axis.Regular(60, 0, 120, name="m", label="Leading-jet mass [GeV]"), storage=hist.storage.Weight())
h_dijet = hist.Hist(cat, hist.axis.Regular(60, 0, 2000, name="m", label="Leading dijet mass [GeV]"), storage=hist.storage.Weight())
h_sys = hist.Hist(cat, hist.axis.Regular(60, 0, 5000, name="m", label="All-jet system mass [GeV]"), storage=hist.storage.Weight())

# --- 1D weighted hists (summed over slices) ---
def H_pt():
    return hist.Hist(hist.axis.Regular(60, 0, 2000, name="x", label="pT [GeV]"), storage=hist.storage.Weight())

def H_ht():
    return hist.Hist(hist.axis.Regular(60, 0, 5000, name="x", label="HT [GeV]"), storage=hist.storage.Weight())

h_pt_scout = H_pt(); h_pt_scout_raw = H_pt(); h_pt_genjet = H_pt(); h_pt_genpart = H_pt()
h_ht_branch = H_ht(); h_ht_scout = H_ht(); h_ht_scout_raw = H_ht(); h_ht_genjet = H_ht(); h_ht_genpart = H_ht()

for name in DATASETS:
    files = fileset[name]["files"]              # {path: tree}
    n_orig = slice_n_original(list(files))      # ALL files; cutflow only
    weight = CROSS_SECTIONS[name] / n_orig * LUMI
    to_read = files if not N_FILES else dict(list(files.items())[:N_FILES])

    events = NanoEventsFactory.from_root(
        to_read, mode="dask", schemaclass=BaseSchema,
        metadata={"dataset": name, "weight": weight},
    ).events()
    w = events.metadata["weight"]
    flds = events.fields

    jets = jet_p4(events)            # corrected (post-JEC) scouting jets
    njet = ak.num(jets)
    sj = events["ScoutingPFJet"]

    lazy = {}
    # masses
    lazy["lead_m"] = jets[njet >= 1][:, 0].mass
    two = jets[njet >= 2]
    lazy["dijet_m"] = (two[:, 0] + two[:, 1]).mass
    system = ak.zip({"px": ak.sum(jets.px, axis=1), "py": ak.sum(jets.py, axis=1),
                     "pz": ak.sum(jets.pz, axis=1), "E": ak.sum(jets.energy, axis=1)},
                    with_name="Momentum4D")
    lazy["sys_m"] = system.mass
    # scouting (post-JEC) lead pT + HT
    lazy["pt_scout"] = lead_pt(jets.pt)
    lazy["ht_branch"] = events["HT"]               # event HT from the branch
    lazy["ht_scout"] = ak.sum(jets.pt, axis=1)     # recalculated from jets
    # scouting pre-JEC (raw), if present
    has_raw = "pt_raw" in sj.fields
    if has_raw:
        lazy["pt_scout_raw"] = lead_pt(sj.pt_raw)
        lazy["ht_scout_raw"] = ak.sum(sj.pt_raw, axis=1)
    # GenJet (no branch HT)
    has_gj = "GenJet" in flds
    if has_gj:
        gjpt = events["GenJet"].pt
        lazy["pt_genjet"] = lead_pt(gjpt)
        lazy["ht_genjet"] = ak.sum(gjpt, axis=1)
    # GenPart (no branch HT; full gen record -- not a physical HT, see note)
    has_gp = "GenPart" in flds
    if has_gp:
        gppt = events["GenPart"].pt
        lazy["pt_genpart"] = lead_pt(gppt)
        lazy["ht_genpart"] = ak.sum(gppt, axis=1)

    (vals,) = dask.compute(lazy)

    h_lead.fill(dataset=name, m=ak.to_numpy(vals["lead_m"]), weight=w)
    h_dijet.fill(dataset=name, m=ak.to_numpy(vals["dijet_m"]), weight=w)
    h_sys.fill(dataset=name, m=ak.to_numpy(vals["sys_m"]), weight=w)
    h_pt_scout.fill(x=ak.to_numpy(vals["pt_scout"]), weight=w)
    h_ht_branch.fill(x=ak.to_numpy(vals["ht_branch"]), weight=w)
    h_ht_scout.fill(x=ak.to_numpy(vals["ht_scout"]), weight=w)
    if has_raw:
        h_pt_scout_raw.fill(x=ak.to_numpy(vals["pt_scout_raw"]), weight=w)
        h_ht_scout_raw.fill(x=ak.to_numpy(vals["ht_scout_raw"]), weight=w)
    if has_gj:
        h_pt_genjet.fill(x=ak.to_numpy(vals["pt_genjet"]), weight=w)
        h_ht_genjet.fill(x=ak.to_numpy(vals["ht_genjet"]), weight=w)
    if has_gp:
        h_pt_genpart.fill(x=ak.to_numpy(vals["pt_genpart"]), weight=w)
        h_ht_genpart.fill(x=ak.to_numpy(vals["ht_genpart"]), weight=w)
    print(f"{name}: N_orig={n_orig:,.0f}  xsec={CROSS_SECTIONS[name]:g} pb  w={w:.3e}")

print("done filling")

## Masses

Stacked per-slice contributions (filled) plus the joint QCD total (black step).
With `LUMI = 1.0` the y axis is differential cross section (pb per bin).

In [ ]:
def plot_joint(h, title, logy=True):
    fig, ax = plt.subplots(figsize=(7, 5))
    h.stack("dataset").plot(stack=True, histtype="fill", ax=ax)
    h.project("m").plot(histtype="step", color="black", linewidth=1.5,
                         label="QCD total", ax=ax)
    ax.set_title(title)
    ax.set_ylabel("cross section [pb] / bin" if LUMI == 1.0 else "events")
    if logy:
        ax.set_yscale("log")
    ax.legend(fontsize=6, ncol=2)
    plt.show()


plot_joint(h_lead, "Leading-jet mass (QCD, xsec-weighted)")
plot_joint(h_dijet, "Leading dijet mass (QCD, xsec-weighted)")
plot_joint(h_sys, "All-jet system mass (QCD, xsec-weighted)")

## Leading pT and HT: scouting jets / GenJets / GenParts

Joint QCD totals (summed over slices), overlaid. GenJet/GenPart have no event
HT branch, so only a recalculated (sum-pT) HT is shown for them.

**Caveat:** `GenPart` is the full generator record (all status codes,
including intermediate/duplicate lines), so its "leading pT" and "sum pT" are
not physical observables -- they are shown as requested but need a final-state /
hard-process selection to be meaningful.

In [ ]:
def plot_overlay(items, title, logy=True):
    fig, ax = plt.subplots(figsize=(7, 5))
    for h, lab in items:
        h.plot(ax=ax, histtype="step", label=lab)
    ax.set_title(title)
    ax.set_ylabel("cross section [pb] / bin" if LUMI == 1.0 else "events")
    if logy:
        ax.set_yscale("log")
    ax.legend()
    plt.show()


plot_overlay(
    [(h_pt_scout, "scouting jet"), (h_pt_genjet, "GenJet"), (h_pt_genpart, "GenPart")],
    "Leading pT (QCD, xsec-weighted)",
)

In [ ]:
# Event HT: branch vs recalculated (scouting), and recalculated GenJet HT.
plot_overlay(
    [(h_ht_branch, "scouting HT (branch)"),
     (h_ht_scout, "scouting HT (recalc from jets)"),
     (h_ht_genjet, "GenJet HT (recalc)")],
    "Event HT (QCD, xsec-weighted)",
)

In [ ]:
# GenPart sum-pT shown separately (different scale; not a physical HT).
plot_overlay([(h_ht_genpart, "GenPart sum pT")],
             "GenPart sum pT (not a physical HT)")

## Pre- vs post-JEC (scouting jets)

Raw (pre-JEC, `ScoutingPFJet.pt_raw`) vs corrected (post-JEC, `ScoutingPFJet.pt`)
for the leading-jet pT and the recalculated HT. No gen-jet matching here (that
comes later). Only shown if the slimmed files carry `pt_raw` (corrections on).

In [ ]:
if h_pt_scout_raw.sum().value > 0:
    plot_overlay(
        [(h_pt_scout_raw, "raw (pre-JEC)"), (h_pt_scout, "corrected (post-JEC)")],
        "Leading scouting-jet pT: pre vs post JEC",
    )
    plot_overlay(
        [(h_ht_scout_raw, "raw (pre-JEC)"), (h_ht_scout, "corrected (post-JEC)")],
        "Recalc HT (scouting): pre vs post JEC",
    )
else:
    print("No pt_raw branch -- these slimmed files were made without JEC corrections.")